In [1]:
import os
import json
import requests
import asyncio
from pageindex import PageIndexClient
import pageindex.utils as utils

In [ ]:
PAGEINDEX_API_KEY = ""
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [5]:
pdf_path = r"C:\Users\ttejkiran\OneDrive - Deloitte (O365D)\Documents\Books\2501.12948v2.pdf"

doc_info = pi_client.submit_document(pdf_path)
doc_id = doc_info["doc_id"]

print('Document Submitted:', doc_id)

Document Submitted: pi-cmnvqpq6r0dvi01qsr2xyeuou


In [ ]:
# doc_id = "pi-cmnvqpq6r0dvi01qsr2xyeuou"

In [6]:
status = pi_client.get_document(doc_id)

In [8]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 18

🌲 Raw tree (first node):
{
  "title": "Abstract",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This text discusses the challenge of general reasoning in AI, highlighting the limitations of current LLMs that rely on human annotations. It introduces a novel reinforcement learning (RL) framework that enables LLMs to develop advanced reasoning abilities, such as self-reflection and verification, without human-labeled data. This RL-trained model achieves superior performance on verifiable tasks compared to models trained with supervised learning, and its emergent reasoning patterns can be leveraged to improve smaller models.",
  "text": "# Abstract\n\nGeneral reasoning represents a long-standing and formidable challenge in artificial intelligence. Recent breakthroughs, exemplified by large language models (LLMs) *(Brown et al., 2020; OpenAI, 2023)* and chain-of-thought prompting *(Wei et al., 2022b)*, have achieved considerable success on foundational reas

In [9]:
import time

print(f"Waiting for document {doc_id} to be indexed...")

max_retries = 30 
retry_count = 0

while not pi_client.is_retrieval_ready(doc_id):
    if retry_count >= max_retries:
        print("Timeout: Document processing took too long.")
        break
    
    print(f"Still processing... (Attempt {retry_count + 1}/{max_retries})")
    time.sleep(5)
    retry_count += 1

if pi_client.is_retrieval_ready(doc_id):
    print("Success! Document is ready.")
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    utils.print_tree(tree)
else:
    tree = None

Waiting for document pi-cmnvqpq6r0dvi01qsr2xyeuou to be indexed...
Success! Document is ready.
[{'title': 'Abstract', 'node_id': '0000', 'summary': 'This text discusses the challenge of gen...'},
 {'title': '1 Introduction',
  'node_id': '0001',
  'summary': 'This text discusses the development of r...'},
 {'title': '2 DeepSeek-R1-Zero',
  'node_id': '0002',
  'summary': 'The text details the training of DeepSee...'},
 {'title': '3. DeepSeek-R1',
  'node_id': '0003',
  'summary': 'This text introduces DeepSeek-R1, an enh...'},
 {'title': '4 Experiment',
  'node_id': '0004',
  'summary': 'The text details the experimental evalua...'},
 {'title': '5 Ethics and Safety Statement',
  'node_id': '0005',
  'summary': 'The text addresses the ethical risks of ...'},
 {'title': '6 Conclusion, Limitation, and Future Wor...',
  'node_id': '0006',
  'summary': 'The text introduces DeepSeek-R1-Zero and...'},
 {'title': '7 Author List',
  'node_id': '0007',
  'summary': 'The text presents an author l

In [10]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Abstract  (p.1)
[0001] 1 Introduction  (p.1)
[0002] 2 DeepSeek-R1-Zero  (p.2)
[0003] 3. DeepSeek-R1  (p.6)
[0004] 4 Experiment  (p.8)
[0005] 5 Ethics and Safety Statement  (p.10)
[0006] 6 Conclusion, Limitation, and Future Work  (p.10)
[0007] 7 Author List  (p.11)
[0008] Appendix A Background  (p.13)
[0009] B. Training Details  (p.17)
  └─ [0010] B.1. RL Infrastructure  (p.17)
  └─ [0011] B.2 Reward Model Prompt  (p.18)
  └─ [0012] B.3. Data Recipe  (p.19)
  └─ [0013] Non-Reasoning Data  (p.26)
    └─ [0014] B.4 Hyper-Parameters  (p.34)
    └─ [0015] B.5. Reward Hacking  (p.35)
    └─ [0016] B.6. Ablation Study of Language Consistency Reward  (p.35)
[0017] C. Self-Evolution of DeepSeek-R1-Zero  (p.37)
[0018] D. Evaluation of DeepSeek-R1  (p.39)
  └─ [0019] D.1. Experiment Setup  (p.39)
  └─ [0020] Decontamination  (p.40)
  └─ [0021] Evaluation Prompts  (p.40)
  └─ [0022] Standard Benchmark  (p.41)
  └─ [0023] Potential Risky Dialogue Filtering  (p.46)

In [ ]:
AZURE_OPENAI_API_KEY=''
AZURE_OPENAI_ENDPOINT=''
AZURE_OPENAI_DEPLOYMENT_NAME=''
AZURE_OPENAI_TEXT_MODEL_NAME=''
AZURE_OPENAI_API_VERSION=''

In [12]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    openai_api_key=AZURE_OPENAI_API_KEY,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    deployment_name=AZURE_OPENAI_DEPLOYMENT_NAME,
    model=AZURE_OPENAI_TEXT_MODEL_NAME,
    temperature=0
)

In [13]:
def retrieve_from_pageindex(query, doc_id, top_k=3):
    
    response = pi_client.submit_query(
        doc_id=doc_id,
        query=query
    )

    retrieval_id = response.get("retrieval_id")
    if not retrieval_id:
        return []

    while True:
        retrieval = pi_client.get_retrieval(retrieval_id)
        status = retrieval.get("status")

        if status == "completed":
            break
        elif status == "failed":
            return []

        time.sleep(1)

    nodes = retrieval.get("retrieved_nodes", [])
    contexts = []

    for node in nodes[:top_k]:
        relevant_contents = node.get("relevant_contents", [])
        
        for group in relevant_contents:
            for item in group:
                content = item.get("relevant_content")
                if content:
                    contexts.append(content)

    return contexts

In [14]:
def vectorless_rag(query, doc_id):

    contexts = retrieve_from_pageindex(query, doc_id)

    if not contexts:
        return "No relevant context found."

    combined_context = "\n\n".join(contexts)

    prompt = f"""
    You are a research assistant.
    Answer ONLY using the context below.
    If the answer is not found, say "Not found in document."

    Context:
    {combined_context}

    Question:
    {query}
    """
    response = llm.invoke(prompt)
    return response.content

In [15]:
query = "What is the main contribution of this paper?"

answer = vectorless_rag(query, doc_id)

print("\nFINAL ANSWER:\n")
print(answer)


FINAL ANSWER:

The main contribution of this paper is demonstrating that the reasoning abilities of large language models (LLMs) can be enhanced through pure reinforcement learning (RL) without the need for human-labeled reasoning trajectories. This RL framework enables the emergence of advanced reasoning patterns such as self-reflection, verification, and dynamic strategy adaptation, resulting in models that outperform those trained with conventional supervised learning on human demonstrations in verifiable tasks including mathematics, coding competitions, and STEM fields. Additionally, the paper shows that these emergent reasoning patterns can be used to guide and improve the reasoning capabilities of smaller models.
